# Tutorial 11 — Multimodal representations: CLIP, SigLIP, and cross-modal retrieval

[Lecture notes](../lectures/11_contrastive.md) · [Exercise sheet](../exercises/11_contrastive.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Train a small image-to-text alignment model on generated pixel arrays and evaluate a held-out color-position combination.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Generate paired image and label features

Images are 8×8 RGB arrays with one colored block. Text features explicitly encode color and position; they are not learned language embeddings. Hold out blue-right to probe composition.

Before running: predict the output or the invariant being checked.


In [2]:
colors=np.eye(3); names=["red","green","blue"]
images=[]; text=[]; labels=[]
for c in range(3):
    for side in range(2):
        im=np.zeros((8,8,3)); start=0 if side==0 else 4
        im[2:6,start:start+4]=colors[c]
        images.append(im)
        text.append(np.r_[colors[c],np.eye(2)[side]])
        labels.append(names[c]+(" left" if side==0 else " right"))
images=np.array(images); text=np.array(text)
X=images.reshape(6,-1); X/=np.maximum(np.linalg.norm(X,axis=1,keepdims=True),1e-12)
T=text/np.linalg.norm(text,axis=1,keepdims=True)
train=np.arange(5); test=np.array([5]); tau=.2
W=rng.normal(0,.05,(X.shape[1],T.shape[1]))
def softmax(z):
    z=z-z.max(axis=-1,keepdims=True); e=np.exp(z); return e/e.sum(axis=-1,keepdims=True)
print("image batch",images.shape,"labels",labels,"held out",labels[5])


image batch (6, 8, 8, 3) labels ['red left', 'red right', 'green left', 'green right', 'blue left', 'blue right'] held out blue right


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Train symmetric contrastive alignment

The normalization derivative is included explicitly. Training candidates are the five paired training labels; evaluation uses all six candidates.

Before running: predict the output or the invariant being checked.


In [3]:
losses=[]
for step in range(201):
    Z=X[train]@W; norms=np.maximum(np.linalg.norm(Z,axis=1,keepdims=True),1e-12); U=Z/norms
    S=U@T[train].T/tau; row=softmax(S); col=softmax(S.T)
    loss=-.5*(np.log(np.diag(row)).mean()+np.log(np.diag(col)).mean())
    losses.append(float(loss))
    dS=(row-np.eye(5)+col.T-np.eye(5))/10
    dU=dS@T[train]/tau
    dZ=(dU-U*(dU*U).sum(axis=1,keepdims=True))/norms
    W-=.1*X[train].T@dZ
    if step%50==0: print(step,"loss",loss)
assert losses[-1]<losses[0]


0 loss 2.537682633645926
50 loss 0.08793122872872045
100 loss 0.07990517668694617
150 loss 0.07740843716414633
200 loss 0.07631112357035262


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Evaluate and inspect the compositional gap

A small aligned training set need not recover an unseen combination. Report what actually happens; a failure is useful evidence about this representation.

Before running: predict the output or the invariant being checked.


In [4]:
Z=X@W; U=Z/np.maximum(np.linalg.norm(Z,axis=1,keepdims=True),1e-12)
S=U@T.T/tau
for i in range(6): print(labels[i],"->",labels[int(S[i].argmax())],"held-out" if i==5 else "train")
print("image-to-text top1",np.mean(S.argmax(axis=1)==np.arange(6)))
print("text-to-image top1",np.mean(S.argmax(axis=0)==np.arange(6)))
print("similarity matrix\n",np.round(S,2))
# Uniform logits must give log(N), regardless of temperature.
assert np.isclose(-np.log(softmax(np.zeros((3,3)))[0,0]),np.log(3))


red left -> red left train
red right -> red right train
green left -> green left train
green right -> green right train
blue left -> blue left train
blue right -> blue right held-out
image-to-text top1 1.0
text-to-image top1 0.8333333333333334
similarity matrix
 [[ 3.37  0.34 -0.09 -3.12  0.04 -2.99]
 [ 0.    3.59 -3.47  0.12 -2.04  1.54]
 [-0.21 -3.23  3.27  0.25  0.05 -2.97]
 [-3.48  0.12 -0.06  3.54 -2.28  1.32]
 [-0.51 -1.8  -0.54 -1.83  3.65  2.36]
 [-4.12 -0.07 -2.51  1.53 -2.38  1.67]]


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. Why normalize embeddings before cosine retrieval?
2. What happens when two captions in a batch are both valid for one image?
3. Can a CLIP similarity score be read as a calibrated probability?


## Extension and submission

Run `extensions/hf_vision.py --mode clip` for actual pretrained CLIP retrieval on the supplied generated images.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
